# Three-Group Unsupervised Wellbeing Analysis

This notebook mirrors `unsupervised_3group_wellbeing_analysis.py` step by step so you can see what happens in each cell.

Important scientific note: clustering is unsupervised. The algorithms do not use `true_label` during fitting. Known labels are added only afterward to evaluate whether clusters correspond to `normal`, `sleepy`, and `stress` states.


In [ ]:
from pathlib import Path
import os
import warnings
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report

if not os.environ.get('LOKY_MAX_CPU_COUNT'):
    os.environ['LOKY_MAX_CPU_COUNT'] = str(os.cpu_count() or 1)
warnings.filterwarnings('ignore', category=UserWarning)

from Scripts.ML.unsupervised_3group_wellbeing_analysis import (
    TRUE_LABELS,
    build_dataset,
    build_summary,
    coerce_feature_columns,
    dataset_summary_lines,
    evaluate_clustering,
    fit_clustering_models,
    format_table,
    infer_group_column,
    inspect_group_candidates,
    load_inputs,
    metadata_columns,
    preprocess_full,
    rows_per_group_table,
    run_feature_ablation,
    save_visualizations,
    validate_features,
)

DATA_DIR = Path('data/unsupervised')
RESULTS_DIR = Path('results/unsupervised_3class')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MPL_CONFIG_DIR = RESULTS_DIR / '.matplotlib'
MPL_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPL_CONFIG_DIR.resolve()))

RANDOM_STATE = 42
pd.set_option('display.width', 180)
pd.set_option('display.max_columns', 120)

print(f'Data directory: {DATA_DIR.resolve()}')
print(f'Results directory: {RESULTS_DIR.resolve()}')


Data directory: /Users/mosi_dev/Developer/university/nearable-project/data/unsupervised
Results directory: /Users/mosi_dev/Developer/university/nearable-project/results/unsupervised_3class


## 1. Load normal.csv, sleepy.csv, and stress.csv

The `label` column in `sleepy.csv` and `stress.csv`, if present, is treated as metadata. The notebook creates `true_label` separately only for evaluation after clustering.


In [2]:
raw_datasets = load_inputs(DATA_DIR)

for name, df in raw_datasets.items():
    print()
    print(f'{name}.csv: {df.shape[0]} rows x {df.shape[1]} columns')
    print(list(df.columns))
    if 'label' in df.columns:
        print('Existing label column counts, not used for clustering:')
        print(df['label'].value_counts(dropna=False).to_string())



normal.csv: 218 rows x 20 columns
['window_start', 'video_source', 'p_shoulder_mean', 'p_shoulder_std', 'p_shoulder_bad_pct', 'p_head_tilt_mean', 'p_head_tilt_std', 'p_head_tilt_bad_pct', 'p_fhp_mean', 'p_fhp_std', 'p_fhp_bad_pct', 'b_ear_mean', 'b_ear_std', 'b_perclos', 'b_blink_count', 'b_blink_freq', 'b_ibi_mean', 'b_ibi_std', 'HR', 'RMMSD']

sleepy.csv: 213 rows x 21 columns
['window_start', 'video_source', 'p_shoulder_mean', 'p_shoulder_std', 'p_shoulder_bad_pct', 'p_head_tilt_mean', 'p_head_tilt_std', 'p_head_tilt_bad_pct', 'p_fhp_mean', 'p_fhp_std', 'p_fhp_bad_pct', 'b_ear_mean', 'b_ear_std', 'b_perclos', 'b_blink_count', 'b_blink_freq', 'b_ibi_mean', 'b_ibi_std', 'HR', 'RMMSD', 'label']
Existing label column counts, not used for clustering:
label
sleepy    213

stress.csv: 218 rows x 21 columns
['window_start', 'video_source', 'p_shoulder_mean', 'p_shoulder_std', 'p_shoulder_bad_pct', 'p_head_tilt_mean', 'p_head_tilt_std', 'p_head_tilt_bad_pct', 'p_fhp_mean', 'p_fhp_std', 'p_f

## 2. Inspect participant / source columns

This checks columns such as `video_source` and infers the participant/source column used for reporting group counts. It is not used as a model feature.


In [3]:
group_inspection = inspect_group_candidates(raw_datasets)
group_col = infer_group_column(raw_datasets)

print(f'Inferred group column for reporting: {group_col}')
for file_name, columns in group_inspection.items():
    print()
    print(f'{file_name}.csv candidate group columns:')
    if not columns:
        print('  none detected')
    for column, values in columns.items():
        print(f'  {column}: {values}')


Inferred group column for reporting: video_source

normal.csv candidate group columns:
  video_source: ['Alessia', 'Alvaro', 'Beatrice', 'Mo']

sleepy.csv candidate group columns:
  video_source: ['Alessia', 'Alvaro', 'Beatrice', 'Mo']

stress.csv candidate group columns:
  video_source: ['Alessia', 'Alvaro', 'Beatrice', 'Mo']


## 3. Validate numerical features and missing values

Only the listed numerical sensor features are used. Identifier columns such as `video_source`, participant names, filenames, `window_start`, `label`, `true_label`, and IDs are excluded.


In [4]:
features = validate_features(raw_datasets)
datasets = coerce_feature_columns(raw_datasets, features)

print('Selected numerical feature columns:')
print(features)
print()
print('Note: RMMSD is the CSV column name; scientifically it refers to RMSSD.')
print()
print('Missing values in selected features:')
for name, df in datasets.items():
    missing = df.loc[:, features].isna().sum()
    print()
    print(f'{name}.csv')
    print(missing.to_string())


Selected numerical feature columns:
['p_shoulder_mean', 'p_shoulder_std', 'p_shoulder_bad_pct', 'p_head_tilt_mean', 'p_head_tilt_std', 'p_head_tilt_bad_pct', 'p_fhp_mean', 'p_fhp_std', 'p_fhp_bad_pct', 'b_ear_mean', 'b_ear_std', 'b_perclos', 'b_blink_count', 'b_blink_freq', 'b_ibi_mean', 'b_ibi_std', 'HR', 'RMMSD']

Note: RMMSD is the CSV column name; scientifically it refers to RMSSD.

Missing values in selected features:

normal.csv
p_shoulder_mean        0
p_shoulder_std         0
p_shoulder_bad_pct     0
p_head_tilt_mean       0
p_head_tilt_std        0
p_head_tilt_bad_pct    0
p_fhp_mean             0
p_fhp_std              0
p_fhp_bad_pct          0
b_ear_mean             0
b_ear_std              0
b_perclos              0
b_blink_count          0
b_blink_freq           0
b_ibi_mean             0
b_ibi_std              0
HR                     0
RMMSD                  0

sleepy.csv
p_shoulder_mean        0
p_shoulder_std         0
p_shoulder_bad_pct     0
p_head_tilt_mean       0

## 4. Build the combined dataframe

`true_label` is created here for post-clustering evaluation. It is not included in the feature matrix.


In [5]:
full_df = build_dataset(datasets, group_col)
y_true = full_df['true_label'].tolist()

print(f'Combined rows: {len(full_df)}')
print('Class distribution:')
print(full_df['true_label'].value_counts().to_string())
print()
print('Rows per participant/group per true_label:')
if group_col is not None:
    print(format_table(rows_per_group_table(full_df, group_col)))
else:
    print('No clean group column detected.')


Combined rows: 649
Class distribution:
true_label
normal    218
stress    218
sleepy    213

Rows per participant/group per true_label:
video_source  normal  sleepy  stress  All
     Alessia      54      55      55  164
      Alvaro      54      56      55  165
    Beatrice      55      54      54  163
          Mo      55      48      54  157
         All     218     213     218  649


## 5. Preprocess features without labels

The preprocessing pipeline uses median imputation and standard scaling. It receives only sensor features.


In [6]:
X_scaled, preprocess_pipeline = preprocess_full(full_df, features)

print(f'Raw feature matrix shape: {full_df.loc[:, features].shape}')
print(f'Scaled feature matrix shape: {X_scaled.shape}')
print('Preprocessing pipeline:')
print(preprocess_pipeline)


Raw feature matrix shape: (649, 18)
Scaled feature matrix shape: (649, 18)
Preprocessing pipeline:
Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])


## 6. Run 3-group clustering

K-Means, Gaussian Mixture, and Agglomerative clustering are run with 3 clusters/components. Labels are not used during fitting.


In [7]:
cluster_assignments = fit_clustering_models(X_scaled, RANDOM_STATE)

for method_name, cluster_ids in cluster_assignments.items():
    print(f'{method_name} cluster counts:')
    print(pd.Series(cluster_ids).value_counts().sort_index().to_string())
    print()


K-Means cluster counts:
0    138
1    355
2    156

Gaussian Mixture cluster counts:
0    135
1    260
2    254

Agglomerative cluster counts:
0    364
1    159
2    126



  File "/Users/mosi_dev/Developer/university/nearable-project/.venv/lib/python3.11/site-packages/joblib/externals/loky/backend/context.py", line 249, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_darwin()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/mosi_dev/Developer/university/nearable-project/.venv/lib/python3.11/site-packages/joblib/externals/loky/backend/context.py", line 318, in _count_physical_cores_darwin
    return int(cpu_info)
           ^^^^^^^^^^^^^


## 7. Evaluate clusters after fitting

This step compares cluster IDs with `true_label`, maps clusters to labels using majority vote, and computes interpretive mapped metrics. These metrics are for interpretation only; clustering itself was unsupervised.


In [8]:
table_filenames = {
    'K-Means': 'kmeans_cluster_label_table.csv',
    'Gaussian Mixture': 'gmm_cluster_label_table.csv',
    'Agglomerative': 'agglomerative_cluster_label_table.csv',
}
cm_filenames = {
    'K-Means': 'kmeans_confusion_matrix.png',
    'Gaussian Mixture': 'gmm_confusion_matrix.png',
    'Agglomerative': 'agglomerative_confusion_matrix.png',
}

clustering_rows = []
cluster_tables = {}
cluster_predictions = {}

for method_name, cluster_ids in cluster_assignments.items():
    row, table, mapped = evaluate_clustering(
        method_name,
        np.asarray(cluster_ids),
        X_scaled,
        y_true,
        RESULTS_DIR,
        table_filenames[method_name],
        cm_filenames[method_name],
    )
    clustering_rows.append(row)
    cluster_tables[method_name] = table
    cluster_predictions[method_name] = mapped

clustering_results = pd.DataFrame(clustering_rows)
clustering_results.to_csv(RESULTS_DIR / 'clustering_3group_results.csv', index=False)

result_cols = [
    'method',
    'silhouette',
    'davies_bouldin',
    'calinski_harabasz',
    'adjusted_rand_index',
    'normalized_mutual_info',
    'homogeneity',
    'completeness',
    'v_measure',
    'accuracy',
    'balanced_accuracy',
    'macro_f1',
    'weighted_f1',
]
print('3-group clustering metrics:')
print(format_table(clustering_results.loc[:, result_cols]))

for method_name, table in cluster_tables.items():
    print()
    print(f'{method_name} cluster vs true_label table:')
    print(table.to_string())


3-group clustering metrics:
          method  silhouette  davies_bouldin  calinski_harabasz  adjusted_rand_index  normalized_mutual_info  homogeneity  completeness  v_measure  accuracy  balanced_accuracy  macro_f1  weighted_f1
         K-Means    0.360739        1.362237         276.812886             0.426546                0.514415     0.491779      0.539235   0.514415  0.778120           0.777663  0.780995     0.780623
Gaussian Mixture    0.258200        1.653788         209.864589             0.556373                0.542731     0.533262      0.552542   0.542731  0.821263           0.821804  0.814632     0.814719
   Agglomerative    0.354397        1.355392         266.518370             0.423860                0.549083     0.521244      0.580064   0.549083  0.775039           0.774820  0.778846     0.778260

K-Means cluster vs true_label table:
true_label  normal  sleepy  stress mapped_label
cluster                                        
0                0       4     134       s

## 8. Classification reports after majority-vote mapping

Again, these reports are interpretive. The clusters were formed without labels.


In [9]:
for method_name, mapped in cluster_predictions.items():
    print()
    print(f'{method_name} majority-vote mapped classification report:')
    print(classification_report(y_true, mapped, labels=TRUE_LABELS, zero_division=0))



K-Means majority-vote mapped classification report:
              precision    recall  f1-score   support

      normal       0.61      1.00      0.76       218
      sleepy       0.98      0.72      0.83       213
      stress       0.97      0.61      0.75       218

    accuracy                           0.78       649
   macro avg       0.86      0.78      0.78       649
weighted avg       0.85      0.78      0.78       649


Gaussian Mixture majority-vote mapped classification report:
              precision    recall  f1-score   support

      normal       0.83      0.97      0.90       218
      sleepy       0.73      0.89      0.80       213
      stress       0.97      0.60      0.74       218

    accuracy                           0.82       649
   macro avg       0.85      0.82      0.81       649
weighted avg       0.85      0.82      0.81       649


Agglomerative majority-vote mapped classification report:
              precision    recall  f1-score   support

      nor

## 9. PCA and t-SNE visualizations

PCA plots are saved for true labels and all three clustering methods. t-SNE plots are saved for true labels and K-Means clusters.


In [10]:
visualization_status = save_visualizations(
    X_scaled,
    y_true,
    cluster_assignments,
    RESULTS_DIR,
    RANDOM_STATE,
)

print('Visualization status:')
print(visualization_status)
print('Saved plots in results/unsupervised_3class/.')


Visualization status:
{'pca_saved': True, 'tsne_saved': True}
Saved plots in results/unsupervised_3class/.


### Saved visualization previews

![PCA true labels](results/unsupervised_3class/pca_true_labels.png)

![PCA K-Means clusters](results/unsupervised_3class/pca_kmeans_clusters.png)

![PCA GMM clusters](results/unsupervised_3class/pca_gmm_clusters.png)

![PCA Agglomerative clusters](results/unsupervised_3class/pca_agglomerative_clusters.png)

![t-SNE true labels](results/unsupervised_3class/tsne_true_labels.png)

![t-SNE K-Means clusters](results/unsupervised_3class/tsne_kmeans_clusters.png)


## 10. Feature ablation

K-Means and GMM are rerun using posture-only, eye/blink-only, HR/HRV-only, and all-feature groups.


In [11]:
ablation_results = run_feature_ablation(full_df, RANDOM_STATE)
ablation_results.to_csv(RESULTS_DIR / 'feature_ablation_3group.csv', index=False)

ablation_cols = [
    'feature_group',
    'method',
    'n_features',
    'silhouette',
    'davies_bouldin',
    'calinski_harabasz',
    'adjusted_rand_index',
    'normalized_mutual_info',
    'v_measure',
    'accuracy',
    'balanced_accuracy',
    'macro_f1',
    'weighted_f1',
]
print('Feature ablation results:')
print(format_table(ablation_results.loc[:, ablation_cols]))


Feature ablation results:
 feature_group           method  n_features  silhouette  davies_bouldin  calinski_harabasz  adjusted_rand_index  normalized_mutual_info  v_measure  accuracy  balanced_accuracy  macro_f1  weighted_f1
  posture_only          K-Means           9    0.462387        1.177756         430.811149             0.338845                0.364764   0.364764  0.691834           0.691175  0.675748     0.675668
  posture_only Gaussian Mixture           9    0.379226        1.404906         307.413572             0.405019                0.412224   0.412224  0.702619           0.703242  0.670968     0.670754
eye_blink_only          K-Means           7    0.314624        1.184763         341.734564             0.180534                0.229003   0.229003  0.557781           0.553517  0.440293     0.443685
eye_blink_only Gaussian Mixture           7    0.234008        1.398539         307.227568             0.249323                0.263717   0.263717  0.617874           0.617098  0

## 11. Save per-row predictions

This file includes metadata, `true_label`, cluster IDs, and majority-vote mapped labels for each clustering method.


In [12]:
predictions_df = full_df.loc[:, metadata_columns(full_df, features)].copy()
predictions_df['true_label'] = full_df['true_label']
predictions_df['kmeans_cluster'] = cluster_assignments['K-Means']
predictions_df['kmeans_mapped_label'] = cluster_predictions['K-Means']
predictions_df['gmm_cluster'] = cluster_assignments['Gaussian Mixture']
predictions_df['gmm_mapped_label'] = cluster_predictions['Gaussian Mixture']
predictions_df['agglomerative_cluster'] = cluster_assignments['Agglomerative']
predictions_df['agglomerative_mapped_label'] = cluster_predictions['Agglomerative']
predictions_df.to_csv(RESULTS_DIR / 'unsupervised_3group_predictions.csv', index=False)

print(f'Prediction dataframe shape: {predictions_df.shape}')
print(list(predictions_df.columns))
print()
print(predictions_df.head().to_string(index=False))


Prediction dataframe shape: (649, 12)
['source_file', 'source_row', 'window_start', 'video_source', 'original_label', 'true_label', 'kmeans_cluster', 'kmeans_mapped_label', 'gmm_cluster', 'gmm_mapped_label', 'agglomerative_cluster', 'agglomerative_mapped_label']

source_file  source_row  window_start video_source original_label true_label  kmeans_cluster kmeans_mapped_label  gmm_cluster gmm_mapped_label  agglomerative_cluster agglomerative_mapped_label
 normal.csv           0           0.0       Alvaro            NaN     normal               1              normal            2           normal                      0                     normal
 normal.csv           1          10.0       Alvaro            NaN     normal               1              normal            2           normal                      0                     normal
 normal.csv           2          20.0       Alvaro            NaN     normal               1              normal            2           normal               

## 12. Final summary and interpretation

This writes `results/unsupervised_3class/final_unsupervised_3class_summary.txt` and prints the final interpretation.


In [13]:
summary_lines = dataset_summary_lines(
    datasets,
    full_df,
    features,
    group_col,
    group_inspection,
)
summary_text = build_summary(
    summary_lines,
    clustering_results,
    cluster_tables,
    cluster_predictions,
    ablation_results,
    full_df,
    visualization_status,
)
summary_path = RESULTS_DIR / 'final_unsupervised_3class_summary.txt'
summary_path.write_text(summary_text + chr(10), encoding='utf-8')

final_start = summary_text.index('FINAL INTERPRETATION')
print(summary_text[final_start:])
print()
print(f'Saved final summary to: {summary_path.resolve()}')
print(f'Saved 3-class unsupervised outputs to: {RESULTS_DIR.resolve()}')


FINAL INTERPRETATION
The data formed clusters that aligned strongly with the three known states.
Best clustering method: Gaussian Mixture (macro F1=0.815, balanced accuracy=0.822, v-measure=0.543, silhouette=0.258).
Classes mixed together after majority-vote mapping: normal->sleepy: 6, sleepy->normal: 19, sleepy->stress: 4, stress->normal: 23, stress->sleepy: 64
Clusters separated the three states well, but not perfectly.
HR/RMSSD dominance check: HR/HRV-only macro F1 was 0.530 for K-Means and 0.813 for GMM.
K-Means: best feature group was all_features (macro F1=0.781, balanced accuracy=0.778, v-measure=0.514); all-feature fusion improved over the best single-modality group. HR/HRV-only macro F1=0.530.
Gaussian Mixture: best feature group was all_features (macro F1=0.815, balanced accuracy=0.822, v-measure=0.543); all-feature fusion improved over the best single-modality group. HR/HRV-only macro F1=0.813.
This is proof-of-concept only because the data comes from 4 participants; more pa